# AquaTrack: fine-tune YOLO11n-pose on **all** SwimXYZ freestyle videos

All freestyle videos = 2,880 videos (5 camera views × 4 swimmers × 144), about **79 GB**, 301 frames each at 1080p.
We don't keep the videos. Each one is downloaded, a few frames are pulled out with their labels, and the video is deleted.

The work is split in two parts, so the free GPU time only goes to training:

| Part | Runtime | What it does | Saved in Drive |
|---|---|---|---|
| **A. Build dataset** | CPU (no GPU needed) | downloads all videos, makes ~35k training images | `MyDrive/aquatrack_swim_full/shards/` (20 zips, ~3 GB) |
| **B. Train** | T4 GPU | trains yolo11n-pose | `MyDrive/aquatrack_swim_full/runs/` |

Both parts can be stopped and restarted: Part A skips finished shards, Part B resumes from the last epoch.
Keep this tab open while it runs (free Colab disconnects idle tabs).

## 0. Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/aquatrack_swim_full')
SHARDS = DRIVE / 'shards'
SHARDS.mkdir(parents=True, exist_ok=True)

!pip install -q ultralytics==8.4.173 requests

### Settings

In [ ]:
CFG = dict(
    VIEWS=["Side_underwater", "Side_water_level", "Side_above_water", "Front", "Aerial"],  # all 5 views
    VAL_SWIMMER="Swimmer_Skin_0,75_Muscle_8",  # 1 of the 4 swimmers is kept for validation only
    FRAMES_PER_VIDEO=8,       # training videos: frames per video, spread evenly
    VAL_FRAMES_PER_VIDEO=2,   # validation videos: fewer, so each epoch's check stays quick
    ZOOM_CROPS=True,          # also add a zoomed-in crop of each frame
    MIN_VISIBLE=8,            # skip frames with fewer joints inside the image
    DOWNLOAD_THREADS=3,       # videos downloaded at the same time
)
# If you change these after shards exist, delete MyDrive/aquatrack_swim_full/shards/ so they get rebuilt.

### Dataset code

In [ ]:
%%writefile /content/swim_build.py
"""Build a YOLO Pose dataset from every SwimXYZ freestyle video, in resumable shards.

One shard = one camera view x one swimmer (5 views x 4 swimmers = 20 shards).
Each finished shard is saved as a zip in Google Drive, so a disconnect only loses the shard in progress.
"""
import io
import random
import shutil
import threading
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import requests

LABEL_ZIP = "https://zenodo.org/api/records/8399376/files/Freestyle_labels.zip/content"
PART2_ZIP = "https://zenodo.org/api/records/8402031/files/Freestyle_part2.zip/content"  # side views
# Aerial and Front views. Record id from the SwimXYZ project page ("Freestyle videos (part 1)"):
# https://g-fiche.github.io/research-pages/swimxyz/
PART1_RECORD = 8402009

ALL_VIEWS = ["Side_underwater", "Side_water_level", "Side_above_water", "Front", "Aerial"]
PART1_VIEWS = {"Front", "Aerial"}

# SwimXYZ "COCO" labels use OpenPose's 18-joint order; this maps them to COCO-17 (YOLO Pose)
TO_COCO17 = [0, 15, 14, 17, 16, 5, 2, 6, 3, 7, 4, 11, 8, 12, 9, 13, 10]
FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
Y_SHIFT = 6  # SwimXYZ y is measured from the bottom; found on the side views by overlaying labels


# ---------- reading single files out of the big remote zips ----------

class HttpFile(io.RawIOBase):
    """A seekable file over HTTP Range requests, so zipfile can pull single members."""

    def __init__(self, url):
        self.session = requests.Session()
        r = self._request("head", url, allow_redirects=True)
        self.url = r.url
        self.size = int(r.headers["Content-Length"])
        self.pos = 0
        self.downloaded = 0

    def _request(self, method, url, **kw):
        for attempt in range(8):
            try:
                r = self.session.request(method, url, timeout=120, **kw)
                if r.status_code == 429:  # Zenodo rate limit: wait and try again
                    wait = int(r.headers.get("Retry-After", 60))
                    print(f"  rate limited, waiting {wait}s")
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                return r
            except requests.RequestException:
                if attempt == 7:
                    raise
                time.sleep(2 ** attempt)
        raise RuntimeError(f"Gave up on {url}")

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        if n == 0 or self.pos >= self.size:
            return b""
        end = min(self.pos + n, self.size) - 1
        r = self._request("get", self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        if r.status_code != 206:
            raise RuntimeError("Server does not support Range requests")
        data = r.content
        self.pos += len(data)
        self.downloaded += len(data)
        return data

    def readinto(self, b):
        data = self.read(len(b))
        b[: len(data)] = data
        return len(data)


def open_remote_zip(url, buffer=1 << 22):
    f = HttpFile(url)
    return zipfile.ZipFile(io.BufferedReader(f, buffer)), f


def find_part1_url():
    """Download link of the zip in the "Freestyle videos (part 1)" Zenodo record."""
    api = f"https://zenodo.org/api/records/{PART1_RECORD}"
    try:
        zips = [f["key"] for f in requests.get(api, timeout=60).json()["files"] if f["key"].endswith(".zip")]
        if zips:
            return f"{api}/files/{zips[0]}/content"
    except Exception as e:
        print("Could not read the Zenodo record, guessing the file name:", e)
    return f"{api}/files/Freestyle_part1.zip/content"


# ---------- labels ----------

def parse_labels(text, height):
    frames = []
    for line in text.splitlines()[1:]:  # first line is the column names
        if not line.strip():
            continue
        v = [float(x.replace(",", ".")) for x in line.strip(";").split(";")]
        pts = [(v[i * 3], height - v[i * 3 + 1] + Y_SHIFT) for i in range(18)]
        frames.append([pts[i] for i in TO_COCO17])
    return frames


def to_yolo_line(pts, w, h, min_visible):
    inside = [(x, y) for x, y in pts if 0 <= x < w and 0 <= y < h]
    if len(inside) < min_visible:
        return None
    xs, ys = [p[0] for p in inside], [p[1] for p in inside]
    pad = 0.08 * max(max(xs) - min(xs), max(ys) - min(ys))
    x1, x2 = max(min(xs) - pad, 0), min(max(xs) + pad, w - 1)
    y1, y2 = max(min(ys) - pad, 0), min(max(ys) + pad, h - 1)
    parts = [0, (x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h]
    for x, y in pts:
        parts += [x / w, y / h, 2] if (0 <= x < w and 0 <= y < h) else [0, 0, 0]
    return " ".join(f"{p:.6f}" if isinstance(p, float) else str(p) for p in parts)


def zoom_crop(frame, pts, rng):
    """Crop so the swimmer fills 50-90% of the width (16:9), like real close-up footage."""
    h, w = frame.shape[:2]
    inside = [(x, y) for x, y in pts if 0 <= x < w and 0 <= y < h]
    xs, ys = [p[0] for p in inside], [p[1] for p in inside]
    body_w = max(max(xs) - min(xs), 1)
    cw = min(int(body_w / rng.uniform(0.5, 0.9)), w)
    ch = min(int(cw * 9 / 16), h)
    cx = (min(xs) + max(xs)) / 2 + rng.uniform(-0.1, 0.1) * cw
    cy = (min(ys) + max(ys)) / 2 + rng.uniform(-0.15, 0.15) * ch
    x0 = int(min(max(cx - cw / 2, 0), w - cw))
    y0 = int(min(max(cy - ch / 2, 0), h - ch))
    return frame[y0:y0 + ch, x0:x0 + cw], [(x - x0, y - y0) for x, y in pts]


def save_image(path, img, max_width=960):
    """Save as JPG, shrinking anything wider than max_width (labels are normalized, so they still fit)."""
    if img.shape[1] > max_width:
        img = cv2.resize(img, (max_width, round(img.shape[0] * max_width / img.shape[1])))
    cv2.imwrite(str(path), img, [cv2.IMWRITE_JPEG_QUALITY, 90])


def frames_from_video(video_path, labels_text, out_dir, split, stem, n_frames, zoom, min_visible, seed):
    """Sample n_frames labeled frames from one video into out_dir/images|labels/split. Returns image count."""
    rng = random.Random(seed)
    cap = cv2.VideoCapture(str(video_path))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    labels = parse_labels(labels_text, h)
    n = min(len(labels), int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    img_dir, lbl_dir = out_dir / "images" / split, out_dir / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    # spread frames evenly over the video (neighboring 60fps frames are near-duplicates)
    picks = sorted({min(n - 1, int((i + rng.random()) * n / n_frames)) for i in range(n_frames)}) if n else []
    count = 0
    for fi in picks:
        line = to_yolo_line(labels[fi], w, h, min_visible)
        if line is None:
            continue
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, frame = cap.read()
        if not ok:
            continue
        name = f"{stem}_f{fi:03d}"
        save_image(img_dir / f"{name}.jpg", frame)
        (lbl_dir / f"{name}.txt").write_text(line + "\n")
        count += 1
        if zoom:
            crop, crop_pts = zoom_crop(frame, labels[fi], rng)
            crop_line = to_yolo_line(crop_pts, crop.shape[1], crop.shape[0], min_visible)
            if crop_line is not None:
                save_image(img_dir / f"{name}_zoom.jpg", crop)
                (lbl_dir / f"{name}_zoom.txt").write_text(crop_line + "\n")
                count += 1
    cap.release()
    return count


# ---------- shards ----------

def list_videos(views, part1_url):
    """Return {(view, swimmer): [video names]} and which zip URL each view lives in."""
    urls = {v: (part1_url if v in PART1_VIEWS else PART2_ZIP) for v in views}
    shards = {}
    for url in dict.fromkeys(urls.values()):
        z, _ = open_remote_zip(url)
        for name in z.namelist():
            view = name.split("/")[0]
            if name.endswith(".webm") and view in views and urls[view] == url:
                shards.setdefault((view, name.split("/")[1]), []).append(name)
    return {k: sorted(v) for k, v in sorted(shards.items())}, urls


_local = threading.local()


def _zips(video_url):
    """Each download thread keeps its own open zips (zipfile handles aren't thread-safe)."""
    if not hasattr(_local, "zips"):
        _local.zips = {}
    if video_url not in _local.zips:
        _local.zips[video_url] = open_remote_zip(video_url)[0]
    if "labels" not in _local.zips:
        _local.zips["labels"] = open_remote_zip(LABEL_ZIP, 1 << 16)[0]
    return _local.zips[video_url], _local.zips["labels"]


def build_shard(view, swimmer, videos, video_url, cfg, work_dir, drive_dir, limit=None):
    """Download, sample and zip one shard. Skips shards already saved in Drive."""
    shard = f"{view}__{swimmer}".replace(",", "_")
    final = drive_dir / f"{shard}.zip"
    if final.exists() and limit is None:
        print(f"{shard}: already in Drive, skipping")
        return
    split = "val" if swimmer == cfg["VAL_SWIMMER"] else "train"
    n_frames = cfg["FRAMES_PER_VIDEO"] if split == "train" else cfg["VAL_FRAMES_PER_VIDEO"]
    out = work_dir / shard
    shutil.rmtree(out, ignore_errors=True)
    tmp = work_dir / "videos"
    tmp.mkdir(parents=True, exist_ok=True)
    videos = videos[:limit] if limit else videos

    def one(k_name):
        k, name = k_name
        vzip, lzip = _zips(video_url)
        local = tmp / f"{shard}_{k:03d}.webm"
        local.write_bytes(vzip.read(name))
        text = lzip.read("Freestyle/" + name[: -len(".webm")] + "/COCO/2D_cam.txt").decode()
        try:
            return frames_from_video(local, text, out, split, f"{shard}_{k:03d}", n_frames,
                                     cfg["ZOOM_CROPS"], cfg["MIN_VISIBLE"], seed=k)
        finally:
            local.unlink(missing_ok=True)  # don't keep the video, only the frames

    t0, images = time.time(), 0
    with ThreadPoolExecutor(cfg["DOWNLOAD_THREADS"]) as pool:
        for done, count in enumerate(pool.map(one, enumerate(videos)), 1):
            images += count
            if done % 20 == 0 or done == len(videos):
                print(f"  {shard}: {done}/{len(videos)} videos, {images} images, {time.time() - t0:.0f}s")
    if limit is None:
        shutil.make_archive(str(final.with_suffix("")), "zip", out)
        shutil.rmtree(out)
        print(f"{shard}: saved to Drive ({split}, {images} images)")
    return out


def assemble(drive_dir, dataset_dir):
    """Unzip every shard into one dataset folder and write data.yaml for this machine."""
    shutil.rmtree(dataset_dir, ignore_errors=True)
    shards = sorted(drive_dir.glob("*.zip"))
    for z in shards:
        shutil.unpack_archive(z, dataset_dir)
    (dataset_dir / "data.yaml").write_text(
        f"path: {dataset_dir}\n"
        "train: images/train\n"
        "val: images/val\n"
        "kpt_shape: [17, 3]\n"
        f"flip_idx: {FLIP_IDX}\n"
        "names:\n  0: person\n"
    )
    for split in ("train", "val"):
        print(split, len(list((dataset_dir / "images" / split).glob("*.jpg"))), "images")
    print(len(shards), "shards")

In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import swim_build as sb
importlib.reload(sb)

# Part A: build the dataset (CPU runtime is fine)

### A1. Find the video zips and list every freestyle video

In [ ]:
PART1_URL = sb.find_part1_url()   # Aerial + Front videos (Zenodo record 8402009)
print('Part 1:', PART1_URL)

shards, urls = sb.list_videos(CFG['VIEWS'], PART1_URL)
for (view, swimmer), vids in shards.items():
    print(f'{view:18s} {swimmer:28s} {len(vids)} videos')
print('total', sum(len(v) for v in shards.values()), 'videos')

### A2. Check the labels on every camera view first (important)

The label fix (`Y_SHIFT`, y measured from the bottom) was only checked on the side views before.
This makes 2 videos per view and draws the joints. **Every view should show green dots on the swimmer.**
If Front or Aerial look wrong (dots flipped or off the body), stop and tell Claude before running A3.

In [ ]:
import cv2, matplotlib.pyplot as plt, shutil
PREVIEW = Path('/content/preview')
shutil.rmtree(PREVIEW, ignore_errors=True)

for view in CFG['VIEWS']:
    key = next(k for k in shards if k[0] == view)
    sb.build_shard(*key, shards[key], urls[view], CFG, PREVIEW, SHARDS, limit=2)

for view in CFG['VIEWS']:
    imgs = sorted(PREVIEW.glob(f'{view}__*/images/*/*.jpg'))[:4]
    fig, axes = plt.subplots(1, len(imgs), figsize=(4.5 * len(imgs), 3.2))
    for ax, p in zip(axes, imgs):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        v = [float(x) for x in (p.parent.parent.parent / 'labels' / p.parent.name / (p.stem + '.txt')).read_text().split()]
        ax.imshow(img)
        for i in range(17):
            x, y, vis = v[5 + i * 3: 8 + i * 3]
            if vis > 0:
                ax.plot(x * w, y * h, 'o', color='lime', markersize=3)
        ax.axis('off')
    fig.suptitle(view)
    plt.show()

### A3. Build all 20 shards

This is the long step (it downloads ~79 GB; how long depends on Zenodo's speed, likely a few hours).
If Colab disconnects, just run Setup, A1 and this cell again: finished shards are skipped.

In [ ]:
WORK = Path('/content/work')
for (view, swimmer), vids in shards.items():
    sb.build_shard(view, swimmer, vids, urls[view], CFG, WORK, SHARDS)
print('All shards done:', len(list(SHARDS.glob('*.zip'))), 'of', len(shards))

# Part B: train (switch to GPU first)

**Runtime > Change runtime type > T4 GPU**, then run **0. Setup** and the cells below.
If Colab disconnects mid-training, do the same: B2 notices the saved checkpoint and continues.

In [ ]:
!nvidia-smi

### B1. Unzip the shards onto the fast local disk

In [ ]:
DATASET = Path('/content/dataset')
sb.assemble(SHARDS, DATASET)

### B2. Train (or resume)

Starts from the pretrained `yolo11n-pose.pt`, like the Mac run. Checkpoints go to Drive every epoch.

In [ ]:
import torch
from ultralytics import YOLO

BASE_MODEL = 'yolo11n-pose.pt'  # 'yolo11s-pose.pt' is bigger and more accurate, but 2-3x slower
RUN_NAME = 'swim_full'          # give each new model its own name, e.g. 'swim_full_s'
BATCH = 64                      # use 32 if you get "CUDA out of memory" (likely with yolo11s-pose)

RUN = DRIVE / 'runs' / RUN_NAME
last = RUN / 'weights' / 'last.pt'

if not torch.cuda.is_available():
    print('No GPU in this session. Choose Runtime > Change runtime type > T4 GPU, then run 0. Setup, B1 and B2 again.')
    print('(B3 and B4 work without a GPU.)')
elif last.exists():
    print('Resuming from', last)
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:  # Ultralytics refuses to resume a run that already finished
        print('Nothing to resume:', e)
        print('Training already finished. Run B3 to see the results.')
else:
    YOLO(BASE_MODEL).train(
        data=str(DATASET / 'data.yaml'),
        epochs=40,
        imgsz=640,
        batch=BATCH,
        device=0,
        project=str(DRIVE / 'runs'),
        name=RUN_NAME,
        exist_ok=True,
        freeze=10,      # same as the Mac run: keeps more of what the model already knew
        patience=10,
        save_period=1,
    )

### B3. Results vs the Mac model

In [ ]:
import pandas as pd
df = pd.read_csv(RUN / 'results.csv')
df.columns = df.columns.str.strip()
print(df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50(P)', 'metrics/mAP50-95(P)']].tail(5).to_string(index=False))
print('\nMac swim_zoom (544 images, side views only): pose mAP50 0.786, pose mAP50-95 0.227')
print('Note: this validation set covers all 5 views, so the numbers are not a perfect like-for-like comparison.')
print('Best model:', RUN / 'weights' / 'best.pt')

### B4. Try it on your own photo or video (upload with the folder icon on the left)

In [ ]:
import cv2, matplotlib.pyplot as plt
from ultralytics import YOLO
RUN = DRIVE / 'runs' / 'swim_full'   # change to your run's name if you trained another one
best = YOLO(str(RUN / 'weights' / 'best.pt'))
res = best.predict('your_photo.jpg', conf=0.25)   # change the file name
plt.figure(figsize=(10, 6)); plt.imshow(cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

`best.pt` is in Drive at `MyDrive/aquatrack_swim_full/runs/swim_full/weights/best.pt`. Download it and on the Mac run:

```bash
python pose_photo.py images/person_swimming2.jpeg --model best.pt
```